# Lesson 22: Compare ACT, Diffusion Policy, and SmolVLA

[Previous: Lesson 21](../21-data-quality/21-data-quality.ipynb) · [Course README](../../README.md) · [Course index](../index.ipynb)

This is the reference notebook. Its pair is [22-comparison.ipynb](22-comparison.ipynb). Run the cells in order. The learner copy stops at the first exercise that you have not written. The reference copy has all answers.

**Result of this lesson.** You will adapt three kinds of robot policy (ACT, Diffusion Policy, and SmolVLA) on the same task and the same training data, `bowl_plate` (put the bowl on the plate). You will save and reload the three adapted checkpoints, freeze a new comparison recipe, and compare the adapted policies on reserved start states. You will report four things: **task success**, **speed** (model-call latency, and simulated time against wall time), **training cost** (measured in your own adaptation runs), and **GPU memory**. You will also report how the pretraining of the three recipes differs. You will use your own functions to turn the raw numbers into a result with a stated uncertainty.

**Time.** About two hours for reading and coding. The full experiment is separate. Each of the three adaptation runs stops at a deadline of 1500 s (the course limit for one training experiment is 7200 s).

**Evidence boundary.** The small examples in this notebook use made-up numbers. They check your code. They say nothing about which policy is better. The cells with an *experiment* tag run real checkpoints, real training, and the real simulator. Only those cells produce measurements. This notebook contains no measured results.

## 1. What a fair comparison needs

You have trained or adapted three policies in earlier lessons. Now you want to know which one to use. A comparison is useful only if it is fair, and it must say what it does not show.

**Same task and same adaptation.** All three policies are adapted on the same training split of the same task, with the same seed, batch size, number of updates, optimizer settings, and deadline. They are then judged on the same simulator, the same start states, the same reset seeds, and the same time limit of 300 ticks (15 s at 20 Hz). The task prompt `put the bowl on the plate` is given to SmolVLA. ACT and Diffusion Policy do not read language: they learned one task.

**Every attempt counts.** One attempt is one start state with one seed. A timeout, a policy error, and an infrastructure error all count as failure. You never remove a failed attempt from the count.

**Paired cases.** Each policy attempts the same cases. A *case* is a start state with its reset seed. You compare the policies case by case. A case where only one policy succeeds is more informative than two overall rates.

**Practical recipes, not pure architectures.** The three policies do not start from the same point. ACT starts from a 200-update local checkpoint with no pretraining. Diffusion Policy starts from a 2000-update local checkpoint with an ImageNet vision encoder. SmolVLA starts from a 5000-update local checkpoint whose vision-language model (VLM) comes from the pinned SmolVLM2 weights. The comparison therefore tests *three prepared recipes*. It does not tell you that one architecture is better in general. The numbers in the recipe file `configs/experiments/comparison.yaml` are settings that someone chose. They are not results.

**Published and local SmolVLA are different things.** The SmolVLA in this comparison is the *local learner*: its VLM comes from the pinned SmolVLM2-500M weights (pretrained by others on image and text data, not on this robot), and its action expert was trained locally on the `bowl_plate` training split. The published policy `smolvla-libero` that Lessons 16–20 used was trained by others on much more LIBERO data. It is not in this comparison, and its numbers must never be mixed with the learner's.

**Every start checkpoint has seen this training split.** Adaptation here means *continuing* to train on the same split under one protocol, with a lower learning rate. It is not transfer to a new task. What you can measure is whether more updates under one protocol change the held-out loss, the development-state outcomes, and the final outcomes. A held-out loss that does not fall is a valid result.

| | ACT | Diffusion Policy | SmolVLA |
| --- | --- | --- | --- |
| Reads language? | no | no | yes (the prompt) |
| How one chunk of actions is made | one forward pass | a loop of denoising passes | one pass over images and text, then a loop of flow steps |
| Actions in one chunk | `chunk_size` | `horizon` | `chunk_size` |
| Actions executed per call, `n` | `n_action_steps` | `n_action_steps` | `n_action_steps` |
| Pretraining in the recipe | none | ImageNet ResNet-18 encoder | SmolVLM2 VLM (no robot data) |

The experiment prints the real values of `n`, the number of start steps, the number of adaptation updates, and the pretraining text for each method from the recipe.

**What you will measure.**

| Quantity | Meaning | Unit |
| --- | --- | --- |
| success | successes divided by attempts | fraction |
| inference latency | wall time of one model call that makes a chunk | seconds |
| speed | model-call time (wall), episode wall time, and simulated time | seconds |
| training cost | updates, samples seen, and time per update in your adaptation run | updates, samples, seconds |
| memory | peak PyTorch GPU memory in this process | MiB |

## 2. Inference work: why one call is not one action

A policy does not run once for each tick. It makes a *chunk* of actions in one *call*, and the controller executes $n$ of them, one per tick. Then it calls the policy again. Over one episode:

$$C\approx\frac{T}{n}$$

Where:
* $n$ - the number of executed actions per call (`n_action_steps`).
* $T$ - the number of ticks in an episode.
* $C$ - the number of policy calls.
* $L_c$ - the wall time of call $c$, in seconds (used below).

* **ACT** runs one forward pass of a transformer for each call. The number of network passes in one call is 1.
* **Diffusion Policy** first encodes the observations once. Then it runs the denoising network $K$ times, where $K$ is `num_inference_steps`. Each pass removes some noise from the action chunk. The recipe sets $K=20$. More steps cost more time. Fewer steps cost less time and can lower the action quality.
* **SmolVLA** encodes images and text with a vision-language model (VLM) once for each call. Then the action expert runs $K$ flow steps. The library default is $K=10$. The same rule applies: more steps cost more time.

<svg xmlns="http://www.w3.org/2000/svg" width="680" height="190" viewBox="0 0 680 190" role="img" aria-label="Work in one policy call for three policies">
  <style>text{font-family:sans-serif;font-size:12px} .b{fill:#dbeafe;stroke:#1d4ed8} .l{fill:#fef9c3;stroke:#a16207} .g{fill:#dcfce7;stroke:#15803d}</style>
  <text x="5" y="16" font-weight="bold">ACT</text>
  <rect class="b" x="60" y="4" width="120" height="20"/><text x="66" y="18">encode + decode once</text>
  <text x="5" y="76" font-weight="bold">Diffusion</text>
  <rect class="b" x="60" y="62" width="70" height="20"/><text x="66" y="76">encode obs</text>
  <rect class="l" x="135" y="62" width="40" height="20"/><text x="142" y="76">pass 1</text>
  <rect class="l" x="180" y="62" width="40" height="20"/><text x="187" y="76">pass 2</text>
  <text x="228" y="76">... K passes (K = 20 in the recipe)</text>
  <text x="5" y="136" font-weight="bold">SmolVLA</text>
  <rect class="g" x="60" y="122" width="110" height="20"/><text x="66" y="136">VLM: images + text</text>
  <rect class="l" x="175" y="122" width="40" height="20"/><text x="182" y="136">step 1</text>
  <rect class="l" x="220" y="122" width="40" height="20"/><text x="227" y="136">step 2</text>
  <text x="268" y="136">... K flow steps (library default K = 10)</text>
  <text x="5" y="180">Each call makes one chunk. The controller then executes n actions, one per 50 ms tick.</text>
</svg>

```text
ACT:       [encode + decode] ........................................ 1 pass per call
Diffusion: [encode obs] [pass 1] [pass 2] ... [pass K] .............. 1 + K passes per call
SmolVLA:   [VLM: images + text] [step 1] ... [step K] ............... 1 + K passes per call
Each call makes one chunk. The controller executes n actions, one per tick.
```

**Work per executed action.** If one call costs $1+K$ network passes, then one executed action costs on average:

$$\text{passes per action}=\frac{1+K}{n}$$

Where $K$ is the number of denoising or flow steps per call ($K=0$ for ACT) and $n$ the executed actions per call. Take Diffusion with $K=20$ and $n=8$: $(1+20)/8=2.625$ passes per action. Take ACT with $n=10$: $1/10=0.1$ passes per action. A pass of a different network has a different cost, so you cannot compare pass counts across models. The count tells you how the cost changes when you change `n` or `K` for *one* model: doubling `n` halves the work per action, and doubling `K` almost doubles the work per call.

**What $n$ changes.** A larger $n$ means fewer calls, and a lower average cost per tick. It also means that the robot runs longer without a new observation. A smaller $n$ makes the robot react faster, and costs more calls.

## 3. Setup

The first code cell finds the workspace and sets offline mode. Nothing downloads in this lesson. The second code cell has all imports.

In [ ]:
import os
from pathlib import Path

WORKSPACE = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "manifests" / "act_references.yaml").is_file()
    ),
    None,
)
if WORKSPACE is None:
    raise FileNotFoundError("Start Jupyter inside the tutorial workspace.")
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
CONFIGS = WORKSPACE / "configs"
DATA = WORKSPACE / ".local-data"

In [ ]:
import json
import math
import time
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from numpy.typing import ArrayLike

from vla_lab.evaluation.comparison import METHODS, load_comparison_config
from vla_lab.teaching.course_comparison import (
    AdaptationRecord,
    AdaptationSettings,
    ComparisonRow,
    Interval,
    RealtimeReport,
    TrainingExposure,
    adapt_policy,
    adapted_recipe,
    case_id,
    checkpoint_heldout_loss,
    count_safetensors_parameters,
    describe_adaptation,
    episode_case_rows,
    episode_time_summary,
    evaluate_development,
    find_completed_data_unit_run,
    freeze_comparison_recipe,
    limit_gpu_memory,
    measure_cuda_use,
    new_run_dir,
    read_jsonl,
    release_gpu_memory,
    require_assets,
    sha256_file,
    training_rate,
    verify_recipe_files,
    write_run_identity,
)
from vla_lab.teaching.evaluation import (
    development_cases,
    evaluate_frozen_method_timed,
    final_cases,
    paired_summary,
)
from vla_lab.teaching.records import LatencySummary, Partition
from vla_lab.timing import LIBERO_CONTROL_HZ, require_libero_rate

## Exercise 22.1: Summarize latency samples

**Purpose.** A mean hides stalls. A robot that is on time 19 times and 100 ms late once has a good mean and a bad tail. Report the mean, the 95th percentile (p95), and the maximum.

**Contract.** `latency_summary(samples_s) -> LatencySummary`

**Input.** `samples_s`: wall times of model calls, in seconds. Shape $(M,)$, finite, never negative. $M$ can be 0.

**Output.** A `LatencySummary` dict (a TypedDict from `vla_lab.teaching.records`; it stays a dict because it is written to JSON files): `count` ($M$), `mean_s`, `p95_s`, `max_s`. If $M=0$ the three values are `None`. A missing measurement stays missing: never report it as 0.

**Definition of p95 (linear interpolation).**

$$h=0.95\,(M-1), \qquad \text{p95}=x_{\lfloor h\rfloor}+\left(h-\lfloor h\rfloor\right)\left(x_{\lceil h\rceil}-x_{\lfloor h\rfloor}\right)$$

Where:
* $x_0\le x_1\le\dots\le x_{M-1}$ - the samples, sorted.
* $M$ - the number of samples.
* $h$ - the fractional position of the 95th percentile in the sorted list.
* $\lfloor h\rfloor$, $\lceil h\rceil$ - $h$ rounded down and up (`lo` and `hi` in the hint).

It is the value below which 95 % of the samples lie, with a straight line between neighbours. A larger $h$ gives a value nearer to the maximum.

**Worked example.** Samples $[0.010, 0.020, 0.030, 0.040, 0.100]$ s. $M=5$, $h=0.95\cdot4=3.8$, $\lfloor h\rfloor=3$, $\lceil h\rceil=4$. So $\text{p95}=0.040+0.8\cdot(0.100-0.040)=0.088$ s. The mean is `0.040` s, the maximum is `0.100` s. The mean is 2.2 times smaller than the p95.

Raise `ValueError` for a sample that is not finite or is negative, or if the input is not one-dimensional.

<details><summary>Hint</summary>

`np.sort` gives the sorted samples. Use `math.floor` and `math.ceil` for `lo` and `hi`. When $h$ is a whole number, `lo == hi` and the formula gives $x_{lo}$. Convert NumPy values with `float(...)`. Convert $M$ with `int(...)`.

</details>

In [ ]:
def latency_summary(samples_s: ArrayLike) -> LatencySummary:
    values = np.asarray(samples_s, dtype=np.float64)
    if values.ndim != 1 or not np.isfinite(values).all() or np.any(values < 0):
        raise ValueError("latency samples must be a finite nonnegative vector")
    if len(values) == 0:
        return {"count": 0, "mean_s": None, "p95_s": None, "max_s": None}
    ordered = np.sort(values)
    position = 0.95 * (len(ordered) - 1)
    low, high = math.floor(position), math.ceil(position)
    p95 = ordered[low] + (position - low) * (ordered[high] - ordered[low])
    return {
        "count": int(len(ordered)),
        "mean_s": float(ordered.mean()),
        "p95_s": float(p95),
        "max_s": float(ordered[-1]),
    }

### Check 22.1

In [ ]:
worked_latency = latency_summary([0.010, 0.020, 0.030, 0.040, 0.100])
assert worked_latency["count"] == 5
assert math.isclose(worked_latency["p95_s"], 0.088), worked_latency
assert math.isclose(worked_latency["mean_s"], 0.040) and math.isclose(
    worked_latency["max_s"], 0.100
)
generator = np.random.default_rng(3)
random_samples = generator.uniform(0.01, 0.2, size=37)
assert math.isclose(
    latency_summary(random_samples)["p95_s"], float(np.quantile(random_samples, 0.95))
)
assert latency_summary([0.05])["p95_s"] == 0.05, "One sample: every statistic is that sample"
empty = latency_summary([])
assert empty == {"count": 0, "mean_s": None, "p95_s": None, "max_s": None}, "Missing stays None"
for bad in ([0.1, float("nan")], [-0.1], [[0.1, 0.2]]):
    try:
        latency_summary(bad)
    except ValueError:
        continue
    raise AssertionError(f"Bad latency samples must raise ValueError: {bad}")
print("22.1 passed:", worked_latency)

## 4. Does the policy keep up with the controller?

**Three clocks.** Keep three times apart. *Simulated time* is the robot's time: $\text{ticks}/20$ seconds. *Model-call wall time* is how long `select_action` or the chunk generator really takes on your computer. *Episode wall time* is how long you wait for a whole episode: the simulator, the rendering, and the model calls together. The three are different numbers, and the experiment reports all three.

LIBERO needs one action every $\text{period}=1/20\ \text{s}=0.05\ \text{s}$. A policy call that takes longer than one period cannot hide inside one tick. What happens depends on how the controller works.

**This simulator waits.** In the course runner the simulator stops while the policy computes. A slow call does not change the physics, so latency does not change the success rate in this experiment. Latency matters when you move the same policy to a real robot, or to a runner that does not wait. The report below describes the cost of the *model*. It does not predict success on a real robot.

Two simple rules show the effect of latency.

1. **Synchronous controller.** The controller stops for the whole call. A call that fits inside one period misses no tick deadline; a longer call misses one deadline for each extra period it spans:
$$m_c=\max\!\left(0,\ \left\lceil\frac{L_c}{P}\right\rceil-1\right), \qquad \mathrm{late\_tick\_fraction}=\frac{\sum_{c=1}^{C}m_c}{n\,C}$$
   A call of 0.12 s with $P=0.05$ s has $L/P=2.4$, so it spans 3 periods and misses 2 deadlines.
2. **Asynchronous controller.** The controller executes the old chunk while the next call runs in the background. A call is hidden if it ends before the old chunk runs out:
$$\mathrm{async\_fit\_fraction}=\frac{1}{C}\sum_{c=1}^{C}\left[L_c\le n\,P\right]$$
   With $n=10$, the budget $nP$ is $0.5$ s.

Also report the mean call time and its cost per tick:

$$\mathrm{mean\_call\_s}=\frac{1}{C}\sum_{c=1}^{C}L_c, \qquad \mathrm{amortized\_s\_per\_tick}=\frac{\mathrm{mean\_call\_s}}{n}$$

Where:
* $P$ - the control period `period_s` $=1/$`control_hz`, in seconds.
* $C$ - the number of calls; $c$ - one call.
* $L_c$ - the wall time of call $c$, in seconds.
* $n$ - executed actions per call.
* $m_c$ - the number of tick deadlines that call $c$ misses.
* $\lceil\cdot\rceil$ - round up; $[\cdot]$ - 1 if the condition is true, else 0.

<svg xmlns="http://www.w3.org/2000/svg" width="680" height="150" viewBox="0 0 680 150" role="img" aria-label="Chunk calls on a 50 ms tick timeline">
  <style>text{font-family:sans-serif;font-size:12px}</style>
  <text x="5" y="16" font-weight="bold">Ticks of 50 ms (n = 10 actions per call)</text>
  <g stroke="#6b7280"><path d="M20 40h600"/><path d="M20 34v12M80 34v12M140 34v12M200 34v12M260 34v12M320 34v12M380 34v12M440 34v12M500 34v12M560 34v12M620 34v12"/></g>
  <text x="20" y="62">tick 0</text><text x="620" y="62">tick 10: next call</text>
  <rect x="20" y="75" width="35" height="18" fill="#bbf7d0" stroke="#15803d"/><text x="62" y="89">call of 30 ms: fits in one period</text>
  <rect x="20" y="100" width="120" height="18" fill="#fecaca" stroke="#b91c1c"/><text x="148" y="114">call of 120 ms: synchronous controller misses 2 deadlines; hidden if asynchronous</text>
  <rect x="20" y="125" width="600" height="18" fill="#fde68a" stroke="#a16207"/><text x="28" y="139">budget n * period = 500 ms</text>
</svg>

```text
tick:   0    1    2    3    4    5    6    7    8    9   10
call:   [30 ms]                                              fits in 1 period
call:   [-------- 120 ms --------]                           spans 3 periods, misses 2
budget: [------------------------ 500 ms --------------------]
```

**Worked example.** $n=10$, `control_hz` $=20$, three calls: $L=[0.03,\,0.12,\,0.60]$ s.

| Call | `L / period` | periods spanned | missed deadlines `m` | fits `n * period = 0.5` s? |
| --- | --- | --- | --- | --- |
| 0.03 s | 0.6 | 1 | 0 | yes |
| 0.12 s | 2.4 | 3 | 2 | yes |
| 0.60 s | 12.0 | 12 | 11 | no |

$\sum m=13$ of $nC=30$ ticks, so `late_tick_fraction` $=13/30=0.4333$. `mean_call_s` $=0.25$ s and `amortized_s_per_tick` $=0.025$ s. `async_fit_fraction` $=2/3$.

**Parameter effects.** A larger $n$ lowers the amortized cost, and lowers the late fraction for the same calls, because the same stall is spread over more ticks. A larger $n$ also raises the async budget $nP$. A faster model lowers every $L_c$.

## Exercise 22.2: Report real-time headroom

**Contract.** `realtime_report(call_latencies_s, n_action_steps, control_hz=LIBERO_CONTROL_HZ) -> RealtimeReport`

* `call_latencies_s`: call times in seconds, shape `(C,)`, `C >= 1`, finite, never negative.
* `n_action_steps`: integer `n >= 1`.
* `control_hz`: must be 20. Call the supplied `require_libero_rate(control_hz)`. It raises `ValueError` for another value, because the checkpoints and the simulator need 20 Hz.

Return a `RealtimeReport` dict (a TypedDict from `vla_lab.teaching.course_comparison`) with the keys `period_s`, `mean_call_s`, `amortized_s_per_tick`, `late_tick_fraction`, and `async_fit_fraction`. Raise `ValueError` for empty or invalid input, and for `n < 1`.

**Float care.** `0.15 * 20` is `3.0000000000000004` in floating point. Its `ceil` is 4, not 3. Subtract `1e-9` before `ceil`: `np.ceil(L * control_hz - 1e-9)`. Also use a small tolerance when you compare with the budget: `L <= n * period + 1e-12`.

<details><summary>Hint</summary>

`periods = np.ceil(values * control_hz - 1e-9)`. `missed = np.maximum(periods - 1, 0)`. `late_tick_fraction = missed.sum() / (n * len(values))`. Convert results with `float(...)`.

</details>

In [ ]:
def realtime_report(
    call_latencies_s: ArrayLike, n_action_steps: int, control_hz: int = LIBERO_CONTROL_HZ
) -> RealtimeReport:
    require_libero_rate(control_hz)
    values = np.asarray(call_latencies_s, dtype=np.float64)
    if values.ndim != 1 or len(values) == 0 or not np.isfinite(values).all() or np.any(values < 0):
        raise ValueError("call latencies must be a nonempty finite nonnegative vector")
    if n_action_steps < 1:
        raise ValueError("n_action_steps must be at least 1")
    period = 1 / control_hz
    missed = np.maximum(np.ceil(values * control_hz - 1e-9) - 1, 0)
    mean_call = float(values.mean())
    return {
        "period_s": period,
        "mean_call_s": mean_call,
        "amortized_s_per_tick": mean_call / n_action_steps,
        "late_tick_fraction": float(missed.sum() / (n_action_steps * len(values))),
        "async_fit_fraction": float(np.mean(values <= n_action_steps * period + 1e-12)),
    }

### Check 22.2

In [ ]:
worked_report = realtime_report([0.03, 0.12, 0.60], 10)
assert math.isclose(worked_report["period_s"], 0.05)
assert math.isclose(worked_report["mean_call_s"], 0.25)
assert math.isclose(worked_report["amortized_s_per_tick"], 0.025)
assert math.isclose(worked_report["late_tick_fraction"], 13 / 30), worked_report
assert math.isclose(worked_report["async_fit_fraction"], 2 / 3), worked_report
edge = realtime_report([0.05, 0.15], 10)
assert math.isclose(edge["late_tick_fraction"], 2 / 20), "0.15 s spans 3 periods, 0.05 s spans 1"
assert realtime_report([0.5], 10)["async_fit_fraction"] == 1.0, "A call equal to the budget fits"
longer_chunk = realtime_report([0.03, 0.12, 0.60], 20)
assert longer_chunk["late_tick_fraction"] < worked_report["late_tick_fraction"], (
    "A larger n spreads the stalls"
)
for bad_call in (
    lambda: realtime_report([], 10),
    lambda: realtime_report([0.1], 0),
    lambda: realtime_report([0.1], 10, control_hz=10),
    lambda: realtime_report([-0.1], 10),
):
    try:
        bad_call()
    except ValueError:
        continue
    raise AssertionError("Invalid input must raise ValueError")
print("22.2 passed:", worked_report)

## 5. Training cost and what it means

Training cost has three parts. They are not the same thing.

* **Steps.** One *step* is one optimizer update. `steps` is a count.
* **Samples seen.** A step uses a batch of $B$ samples. A *sample* is one observation with its action chunk. Two recipes with the same steps and different $B$ have seen different amounts of data.
* **Time.** The first steps include one-time start-up, so the measurement ignores a few warm-up steps and uses the steady-state step time. The projected time assumes that the speed stays the same.

$$\mathrm{samples\_seen}=S\,B, \qquad \mathrm{samples\_per\_second}=\frac{B}{t_s}, \qquad \mathrm{projected\_hours}=\frac{S\,t_s}{3600}$$

Where:
* $S$ - `steps`, the number of optimizer updates.
* $B$ - `batch_size`, samples per step.
* $t_s$ - `seconds_per_step`, the steady-state time of one step.

In this lesson the numbers come from **your own adaptation runs**, not from separate toy jobs. The run has a deadline, so the number of updates it really completed is part of the result.

A recipe with a bigger $B$ can have a slower step and a faster sample rate.

**Worked example.** $S=2000$, $B=8$, $t_s=0.25$ s. Then `samples_seen` $=16000$, `samples_per_second` $=8/0.25=32$, and `projected_hours` $=2000\cdot0.25/3600=0.1389$.

**Budgets and pretraining differ.** A recipe with pretraining starts from weights that already learned something. Its robot-training steps then adapt those weights to the robot. A recipe without pretraining must learn everything from the robot data. So "5000 steps" and "200 steps" are not the same amount of learning, and a comparison of success rates mixes the architecture effect with the budget effect and the pretraining effect. Report all three fields next to the success rate. Do not claim that one architecture wins.

**Memory.** The peak memory of training is higher than the peak of inference. Training stores activations for the backward pass, gradients, and optimizer state. The experiment reports the PyTorch CUDA allocator peak of this process, in MiB ($1\ \text{MiB}=2^{20}=1048576$ bytes), for each adaptation run and for each evaluation. It does not include other processes or memory outside PyTorch.

## Exercise 22.3: Convert a training run into comparable numbers

**Contract.** `training_exposure(steps, batch_size, seconds_per_step) -> TrainingExposure`

* `steps`: integer `>= 1`. `batch_size`: integer `>= 1`. `seconds_per_step`: float `> 0`.
* Return a `TrainingExposure` dict (a TypedDict from `vla_lab.teaching.course_comparison`) with `samples_seen` (`int`), `samples_per_second` (`float`) and `projected_hours` (`float`).
* Raise `ValueError` for a value outside these ranges.

<details><summary>Hint</summary>

Use the three formulas in the worked example. Check the ranges before you divide.

</details>

In [ ]:
def training_exposure(steps: int, batch_size: int, seconds_per_step: float) -> TrainingExposure:
    if (
        steps < 1
        or batch_size < 1
        or not seconds_per_step > 0
        or not math.isfinite(seconds_per_step)
    ):
        raise ValueError("steps and batch_size must be at least 1 and seconds_per_step positive")
    return {
        "samples_seen": steps * batch_size,
        "samples_per_second": batch_size / seconds_per_step,
        "projected_hours": steps * seconds_per_step / 3600,
    }

### Check 22.3

In [ ]:
exposure = training_exposure(2000, 8, 0.25)
assert exposure["samples_seen"] == 16000 and isinstance(exposure["samples_seen"], int)
assert math.isclose(exposure["samples_per_second"], 32.0)
assert math.isclose(exposure["projected_hours"], 2000 * 0.25 / 3600)
small_batch = training_exposure(5000, 1, 0.5)
assert small_batch["samples_seen"] == 5000 and small_batch["samples_per_second"] == 2.0
for bad_call in (
    lambda: training_exposure(0, 8, 0.25),
    lambda: training_exposure(10, 0, 0.25),
    lambda: training_exposure(10, 8, 0.0),
    lambda: training_exposure(10, 8, float("nan")),
):
    try:
        bad_call()
    except ValueError:
        continue
    raise AssertionError("Invalid input must raise ValueError")
print("22.3 passed:", exposure)

## 6. How sure are you? Intervals and paired tests

A success rate from $N$ attempts is a noisy estimate. With 30 attempts, a rate of 0.67 could easily be 0.5 or 0.8 on another set of seeds. Report an interval and a paired test.

### Wilson interval

The Wilson interval is $[\text{center}-\text{half},\ \text{center}+\text{half}]$, kept inside $[0,1]$, with:

$$\hat{p}=\frac{s}{N}, \qquad \text{center}=\frac{\hat{p}+\dfrac{z^2}{2N}}{1+\dfrac{z^2}{N}}, \qquad \text{half}=\frac{z\sqrt{\dfrac{\hat{p}(1-\hat{p})}{N}+\dfrac{z^2}{4N^2}}}{1+\dfrac{z^2}{N}}$$

Where:
* $s$ - the number of successes.
* $N$ - the number of attempts.
* $\hat{p}$ - the observed success rate.
* $z$ - the number of standard deviations that covers 95 % of a normal distribution, $z=1.96$.

It is better than $\hat{p}\pm z\sqrt{\hat{p}(1-\hat{p})/N}$ when $N$ is small or $\hat{p}$ is near 0 or 1, because it never leaves $[0,1]$ and it is not zero-width at $\hat{p}=0$ or $\hat{p}=1$. A larger $z$ gives a wider interval. A larger $N$ gives a narrower one.

**Worked example.** $s=20$, $N=30$: $\hat{p}=0.6667$, $z^2=3.8416$, $1+z^2/N=1.12805$. $\text{center}=(0.6667+0.0640)/1.12805=0.6477$. $\text{half}=1.96\cdot\sqrt{0.007407+0.001067}/1.12805=0.1599$. The interval is $[0.488,\ 0.808]$. It is wide.

## Exercise 22.4: Compute a Wilson interval

**Contract.** `wilson_interval(successes, attempts, z=1.96) -> Interval`. Return an `Interval` dict (a TypedDict from `vla_lab.teaching.course_comparison`) with the keys `low` and `high`, both clipped to `[0, 1]`. Raise `ValueError` if `attempts < 1`, if `successes` is outside `0..attempts`, or if `z <= 0`.

<details><summary>Hint</summary>

Compute `denominator = 1 + z**2 / attempts` once. Then compute `center` and `half`. Use `max(0.0, ...)` and `min(1.0, ...)` for the clip.

</details>

In [ ]:
def wilson_interval(successes: int, attempts: int, z: float = 1.96) -> Interval:
    if attempts < 1 or not 0 <= successes <= attempts or not z > 0:
        raise ValueError("need attempts >= 1, 0 <= successes <= attempts, and z > 0")
    rate = successes / attempts
    denominator = 1 + z**2 / attempts
    center = (rate + z**2 / (2 * attempts)) / denominator
    half = z * math.sqrt(rate * (1 - rate) / attempts + z**2 / (4 * attempts**2)) / denominator
    return {"low": max(0.0, center - half), "high": min(1.0, center + half)}

### Check 22.4

In [ ]:
worked_interval = wilson_interval(20, 30)
assert math.isclose(worked_interval["low"], 0.4878, abs_tol=5e-4), worked_interval
assert math.isclose(worked_interval["high"], 0.8077, abs_tol=5e-4), worked_interval
none = wilson_interval(0, 30)
assert none["low"] == 0.0 and 0.1 < none["high"] < 0.15, (
    "Zero successes still has a nonzero upper bound"
)
everything = wilson_interval(30, 30)
assert everything["high"] == 1.0 and everything["low"] > 0.85
narrow, wide = wilson_interval(200, 300), wilson_interval(20, 30)
assert narrow["high"] - narrow["low"] < wide["high"] - wide["low"], (
    "More attempts give a narrower interval"
)
assert wilson_interval(20, 30, z=2.58)["low"] < worked_interval["low"], (
    "A larger z gives a wider interval"
)
for bad_call in (
    lambda: wilson_interval(1, 0),
    lambda: wilson_interval(31, 30),
    lambda: wilson_interval(1, 3, z=0),
):
    try:
        bad_call()
    except ValueError:
        continue
    raise AssertionError("Invalid input must raise ValueError")
print("22.4 passed:", worked_interval)

### Exact sign test for paired cases

Two policies attempt the same cases. Look only at the cases where they differ; a case where both succeed or both fail says nothing about the difference. If the two policies are equally good, each different case is equally likely to favour either policy, so $a$ counts heads in $n$ fair coin flips. The exact two-sided p-value is:

$$p=\min\!\left(1,\ \frac{2}{2^n}\sum_{i=0}^{k}\binom{n}{i}\right), \qquad n=a+b, \qquad k=\min(a,b)$$

Where:
* $a$ - cases where only the left policy succeeds (`left_only`).
* $b$ - cases where only the right policy succeeds (`right_only`).
* $n$ - the number of cases where the policies differ.
* $\binom{n}{i}$ - the number of ways to choose $i$ items from $n$.

A small $p$ says that a difference this large is rare if the policies are equal. It does not give the size of the difference, and it does not prove that one policy is better in general. If $n=0$ the policies never differ and $p=1$.

**Worked example.** $a=6$, $b=1$: $n=7$, $k=1$. $\binom{7}{0}+\binom{7}{1}=1+7=8$, and $2^7=128$. So $p=2\cdot8/128=0.125$. Six against one still does not reach 0.05. $a=8$, $b=1$: $n=9$, the sum is $1+9=10$, $2^9=512$, $p=20/512=0.0391$.

## Exercise 22.5: Compute the exact sign test

**Contract.** `exact_sign_test(left_only, right_only) -> float`. Raise `ValueError` for a negative count.

<details><summary>Hint</summary>

`math.comb(n, i)` gives $\binom{n}{i}$. Add the terms for `i` from 0 to `k`. Return `1.0` when `n == 0`.

</details>

In [ ]:
def exact_sign_test(left_only: int, right_only: int) -> float:
    if left_only < 0 or right_only < 0:
        raise ValueError("counts must not be negative")
    different = left_only + right_only
    if different == 0:
        return 1.0
    tail = sum(math.comb(different, i) for i in range(min(left_only, right_only) + 1))
    return min(1.0, 2 * tail / 2**different)

### Check 22.5

In [ ]:
assert math.isclose(exact_sign_test(6, 1), 0.125)
assert math.isclose(exact_sign_test(8, 1), 20 / 512)
assert exact_sign_test(1, 6) == exact_sign_test(6, 1), "The test is symmetric"
assert exact_sign_test(0, 0) == 1.0 and exact_sign_test(3, 3) == 1.0
assert exact_sign_test(5, 0) == 0.0625
try:
    exact_sign_test(-1, 2)
except ValueError:
    pass
else:
    raise AssertionError("A negative count must raise ValueError")
print("22.5 passed: p(6, 1) =", exact_sign_test(6, 1))

## 7. Practice the whole pipeline on made-up records

The real run saves one JSON record per attempt in `episodes.jsonl`. The cell below builds records of the same shape from made-up outcomes. The supplied helpers do the record work:

* `episode_case_rows(records, base_seed, prompt, action_steps)` turns saved records into a list of `CaseResult` rows. It checks the seed rule `reset_seed = base_seed + state_id`. It keeps failed and error rows.
* `paired_summary(left, right, expected_case_ids)` pairs rows by case. It checks that state, seed, task, partition, and prompt match. It counts only `success is True`. It refuses missing, repeated, or extra cases. It returns a `PairedSummary`.

Both record types are frozen dataclasses from `vla_lab.teaching.records`; read their fields as attributes:
* `CaseResult` - one attempt on one case: `case_id`, `state_id`, `reset_seed`, `task_id`, `success`, `partition` (a `Partition` enum member: `Partition.DEVELOPMENT` or `Partition.FINAL`), `prompt`, `action_steps` (executed actions per call), and `policy_revision` (which checkpoint ran).
* `PairedSummary` - two policies on the same cases: `cases`, `case_ids`, `left_successes`, `right_successes`, the two policy revisions, `left_success_rate`, `right_success_rate`, `left_only` and `right_only` (cases where only that side succeeded), `paired_difference` ($(\mathrm{left\_successes}-\mathrm{right\_successes})/\text{cases}$), and `failed_attempts_included`. Use `dataclasses.asdict` to write one to JSON.

The outcomes here are **invented to test your code**. They are not results from any policy.

In [ ]:
SYNTHETIC_BASE_SEED = 1000
SYNTHETIC_STATES = list(range(12))
SYNTHETIC_OUTCOMES = {
    "left": [True, True, True, False, True, True, False, True, True, True, False, True],
    "right": [True, False, False, False, True, False, False, True, False, True, False, False],
}


def synthetic_records(method: str) -> list[dict[str, object]]:
    return [
        {
            "task_id": "bowl_plate",
            "initial_state_id": state,
            "partition": Partition.DEVELOPMENT,
            "policy_revision": f"synthetic-{method}",
            "success": outcome,
            "termination_reason": "success" if outcome else "timeout",
            "metrics": {"seed": SYNTHETIC_BASE_SEED + state, "steps": 120 if outcome else 300},
        }
        for state, outcome in zip(SYNTHETIC_STATES, SYNTHETIC_OUTCOMES[method], strict=True)
    ]


synthetic_rows = {
    method: episode_case_rows(
        synthetic_records(method),
        base_seed=SYNTHETIC_BASE_SEED,
        prompt="put the bowl on the plate",
        action_steps=10,
    )
    for method in ("left", "right")
}
synthetic_ids = [case_id(state, SYNTHETIC_BASE_SEED + state) for state in SYNTHETIC_STATES]
synthetic_pair = paired_summary(
    synthetic_rows["left"], synthetic_rows["right"], expected_case_ids=synthetic_ids
)
left_interval = wilson_interval(synthetic_pair.left_successes, synthetic_pair.cases)
right_interval = wilson_interval(synthetic_pair.right_successes, synthetic_pair.cases)
synthetic_p = exact_sign_test(synthetic_pair.left_only, synthetic_pair.right_only)
print("left only / right only:", synthetic_pair.left_only, synthetic_pair.right_only)
print("left rate:", round(synthetic_pair.left_success_rate, 3), "interval:", left_interval)
print("right rate:", round(synthetic_pair.right_success_rate, 3), "interval:", right_interval)
print("exact sign-test p:", round(synthetic_p, 4))
assert synthetic_pair.left_only == 5 and synthetic_pair.right_only == 0
assert math.isclose(synthetic_p, 0.0625)
assert synthetic_pair.failed_attempts_included is True

Look at the made-up result. The left policy wins 5 cases and loses none, yet $p=0.0625$ is above 0.05, and the two Wilson intervals overlap. With twelve cases the evidence is weak even for a clear-looking difference. Real final data has more cases, but the same care applies.

The next cell shows how latency samples become a real-time report. The samples are made up: one slow call every ten calls, like a policy that sometimes reloads a buffer.

In [ ]:
generator = np.random.default_rng(7)
fake_latency = generator.normal(0.040, 0.004, size=50).clip(min=0.001)
fake_latency[::10] = 0.150
for chunk in (5, 10, 20):
    report = realtime_report(fake_latency, chunk)
    print(
        f"n={chunk:2d}",
        "amortized ms/tick:",
        round(1000 * report["amortized_s_per_tick"], 2),
        "| late ticks:",
        round(report["late_tick_fraction"], 3),
        "| async fit:",
        round(report["async_fit_fraction"], 3),
    )
summary = latency_summary(fake_latency)
figure, axis = plt.subplots(figsize=(6, 2.6))
axis.hist(1000 * fake_latency, bins=20)
axis.axvline(1000 * summary["p95_s"], color="tab:red", label="p95")
axis.axvline(1000 * summary["mean_s"], color="tab:green", label="mean")
axis.axvline(1000 / LIBERO_CONTROL_HZ, color="black", linestyle="--", label="50 ms period")
axis.set_xlabel("call time (ms) — made-up samples")
axis.legend()
figure.tight_layout()
plt.close(figure)

## 8. Adapt three policies, freeze, and compare

The experiment groups below run real code with the real simulator and the GPU. They download nothing. The first group checks every local file that the lesson needs, and lists each missing file with the command that creates it. Run the groups in a fresh kernel, in order. Each group finishes before the next group starts, and uses only the output of earlier groups.

| Group | Kind | What it does |
| --- | --- | --- |
| `comparison-setup` | setup | Checks the local files and the GPU. Makes a run directory. Copies the *start* recipe with absolute paths and checks every start hash. Saves the identity of the run: data split, seeds, states, adaptation settings. |
| `comparison-before` | evaluation | Runs each start checkpoint on development states 10–14. |
| `comparison-train-act`, `comparison-train-diffusion`, `comparison-train-smolvla` | training | One group for each policy. Scores the held-out loss, trains under a deadline, scores again, and saves the checkpoint. |
| `comparison-reload` | evaluation | Reloads each saved checkpoint from disk. Checks that it reproduces its held-out loss. Runs it on development states 10–14 and pairs the result with the *before* result. |
| `comparison-freeze` | setup | Writes the new comparison recipe from the three adapted checkpoints, copies it with absolute paths, and checks every hash. |
| `comparison-final-evaluation` | evaluation | After the freeze, runs each frozen policy once on the final states 20–49. Records every attempt, the wall time of every model call, and the memory of each evaluation. |
| `comparison-report` | display | Joins the saved files. Uses your functions to make the table, the intervals, the paired tests, and the plots. |

**The adaptation protocol.** Every policy gets the same protocol: the frozen `bowl_plate` training split (39 episodes), batch size 8, up to 1000 updates, a new AdamW optimizer with learning rate `1e-5` and weight decay `1e-4`, gradient clipping at 10, and seed `20261006`. At least 500 updates must finish, or the group raises an error: a few updates are not an adaptation result. Batches are drawn from one fixed permutation of the training rows for each epoch. Updating only the parameters that each policy marks trainable means that the SmolVLA vision-language model stays frozen.

**Windows come from frame offsets.** A training sample is one row of the dataset with a window of neighbouring rows. ACT uses the action rows `0..19`. Diffusion Policy uses two observation rows and a 16-row action window. SmolVLA uses the action rows `0..49`. The helper turns these row offsets into the timestamp labels of the dataset file (`frame_offset_seconds`, at the 10 FPS *storage label*). It does not use the 20 Hz controller rate to place rows, and it never repeats or skips a row. The offsets that each run used are saved in its `run.json`.

**Deadline.** The training loop tests a deadline of 1500 s before every update, and stops there. It does not wait until the end and test afterwards. The run records whether the update budget (`updates`) or the `deadline` stopped it. The course limit for one training experiment is 7200 s, and the helper refuses a larger deadline.

**Development states.** The course used the development states 0–19 for earlier work. This lesson uses only states 10–14 (the *method* states) for the before and after checks. It never runs a final state before the freeze, and the adaptation code refuses to evaluate one.

**The freeze.** After the three adaptations and the development checks, the freeze group writes the new recipe, resolves every checkpoint to an absolute path, and checks every model hash. After that, the recipe file and its hash are part of the run record. You must not change the recipe, a checkpoint, the prompt, the seeds, or the time limit after you see any final outcome. The final evaluation cell refuses to run if the recipe file changed.

**Budget.** Each adaptation group stops by its 1500 s deadline. The notebook states no measured time, because it has no run to report. Each group prints its own elapsed time.

In [ ]:
require_assets(
    WORKSPACE,
    {
        ".local-data/runs/act-bowl-plate-probe-200/checkpoint-000200/model.safetensors": "the ACT start checkpoint; run `just references-install` (docs/initial-setup.md)",
        ".local-data/runs/diffusion-bowl-plate-reference-2000/checkpoint-002000/model.safetensors": "the Diffusion start checkpoint; run `just diffusion-weights`, then train it as in Lesson 11 (`just train-diffusion`)",
        ".local-data/runs/smolvla-c2-5000-development/checkpoint-005000/model.safetensors": "the SmolVLA learner start checkpoint; train it with `just train-smolvla` (docs/environment.md)",
        ".local-data/models/smolvlm2-500m-video-instruct/model.safetensors": "the pinned SmolVLM2 weights; run `just data-models`",
        ".local-data/hub/lerobot-libero": "the training and validation data; run `just data-training`",
        ".local-data/libero-assets": "the simulator assets; run `just data-demo`",
    },
)
GPU_LIMIT_GB = limit_gpu_memory(48.0)
OUT_DIR = new_run_dir(DATA / "course-results", "comparison")
START_DIR = OUT_DIR / "start-recipe"
START_DIR.mkdir()
START_CONFIG = freeze_comparison_recipe(
    CONFIGS / "experiments/comparison.yaml", workspace=WORKSPACE, output_dir=START_DIR
)
START = load_comparison_config(START_CONFIG, CONFIGS)
START_HASHES = verify_recipe_files(START)
VLM_DIR = Path(START["policies"]["smolvla_learner"]["vlm_dir"])
TASK_SPLIT = yaml.safe_load((CONFIGS / "data/episodes.yaml").read_text())["tasks"]["bowl_plate"]
METHOD_STATES = development_cases("method")
BASE_SEED = START["base_seed"]
ADAPT = AdaptationSettings(
    seed=20261006,
    batch_size=8,
    max_updates=1000,
    min_updates=500,
    max_seconds=1500.0,
    learning_rate=1e-5,
    weight_decay=1e-4,
    heldout_samples=32,
    device="cuda",
)
START_PARAMETERS = {
    method: count_safetensors_parameters(Path(recipe["checkpoint"]) / "model.safetensors")
    for method, recipe in START["policies"].items()
}
write_run_identity(
    OUT_DIR,
    task_alias=START["task_alias"],
    task_prompt=START["task_prompt"],
    dataset_revision=START["dataset_revision"],
    train_split_sha256=START["train_split_sha256"],
    train_episode_ids=TASK_SPLIT["train_episode_ids"],
    validation_episode_ids=TASK_SPLIT["validation_episode_ids"],
    base_seed=BASE_SEED,
    max_steps=START["max_steps"],
    development_states_before_after=METHOD_STATES,
    final_states_reserved=START["state_ids"],
    start_config_sha256=sha256_file(START_CONFIG),
    start_model_sha256=START_HASHES,
    start_parameter_counts=START_PARAMETERS,
    adaptation={
        "seed": ADAPT.seed,
        "batch_size": ADAPT.batch_size,
        "max_updates": ADAPT.max_updates,
        "min_updates": ADAPT.min_updates,
        "max_seconds": ADAPT.max_seconds,
        "learning_rate": ADAPT.learning_rate,
        "weight_decay": ADAPT.weight_decay,
        "heldout_samples": ADAPT.heldout_samples,
    },
    gpu_limit_gb=GPU_LIMIT_GB,
    control_hz=LIBERO_CONTROL_HZ,
)
print("run directory:", OUT_DIR)
for method, recipe in START["policies"].items():
    print(
        method,
        "| start robot steps:",
        recipe["robot_training_steps"],
        "| n_action_steps:",
        recipe["n_action_steps"],
        "| parameters:",
        START_PARAMETERS[method],
        "| pretraining:",
        recipe["pretraining"],
    )

### Before: the start checkpoints on development states 10–14

The first evaluation group runs each *start* checkpoint on the five method states `10..14`, with reset seed `20260930 + state_id`. These are the *before* results. The final states stay unused.

In [ ]:
BEFORE_DIR = OUT_DIR / "dev-before"
before_started = time.monotonic()
before_summaries = {}
for method in METHODS:
    recipe = START["policies"][method]
    before_summaries[method] = evaluate_development(
        method,
        recipe["checkpoint"],
        configs_root=CONFIGS,
        data_root=DATA,
        output_dir=BEFORE_DIR / method,
        state_ids=METHOD_STATES,
        vlm_dir=VLM_DIR,
        inference_steps=START["policies"]["diffusion"]["inference_steps"],
    )
    release_gpu_memory()
    print(
        method,
        "| successes:",
        before_summaries[method]["successes"],
        "of",
        before_summaries[method]["attempts"],
        "| errors:",
        before_summaries[method]["policy_errors"]
        + before_summaries[method]["infrastructure_errors"],
    )
print("before group seconds:", round(time.monotonic() - before_started, 1))

### Adapt: three training groups

Each group below scores the held-out loss of its start checkpoint on the same 32 validation rows (the same noise and time samples every time), trains under the deadline, scores the held-out loss again, and saves a checkpoint with its processors, manifest, and `run.json`. The held-out losses of different methods are *not* comparable with each other: ACT measures an L1 distance, and the other two measure a denoising or flow error. Compare a method only with its own *before* value.

In [ ]:
ADAPT_DIR = OUT_DIR / "adapt"
ADAPT_DIR.mkdir()
ADAPTATIONS: dict[str, AdaptationRecord] = {}
ADAPTATIONS["act"] = adapt_policy(
    "act",
    source=START["policies"]["act"]["checkpoint"],
    source_robot_steps=START["policies"]["act"]["robot_training_steps"],
    output_dir=ADAPT_DIR / "act",
    configs_root=CONFIGS,
    data_root=DATA,
    settings=ADAPT,
)
(ADAPT_DIR / "act.json").write_text(json.dumps(ADAPTATIONS["act"], indent=2, sort_keys=True) + "\n")
print(describe_adaptation(ADAPTATIONS["act"]))

In [ ]:
ADAPTATIONS["diffusion"] = adapt_policy(
    "diffusion",
    source=START["policies"]["diffusion"]["checkpoint"],
    source_robot_steps=START["policies"]["diffusion"]["robot_training_steps"],
    output_dir=ADAPT_DIR / "diffusion",
    configs_root=CONFIGS,
    data_root=DATA,
    settings=ADAPT,
)
(ADAPT_DIR / "diffusion.json").write_text(
    json.dumps(ADAPTATIONS["diffusion"], indent=2, sort_keys=True) + "\n"
)
print(describe_adaptation(ADAPTATIONS["diffusion"]))

In [ ]:
ADAPTATIONS["smolvla_learner"] = adapt_policy(
    "smolvla_learner",
    source=START["policies"]["smolvla_learner"]["checkpoint"],
    source_robot_steps=START["policies"]["smolvla_learner"]["robot_training_steps"],
    output_dir=ADAPT_DIR / "smolvla_learner",
    configs_root=CONFIGS,
    data_root=DATA,
    settings=ADAPT,
    vlm_dir=VLM_DIR,
)
(ADAPT_DIR / "smolvla_learner.json").write_text(
    json.dumps(ADAPTATIONS["smolvla_learner"], indent=2, sort_keys=True) + "\n"
)
print(describe_adaptation(ADAPTATIONS["smolvla_learner"]))

### Reload: did the saved checkpoints keep what they learned?

A checkpoint that cannot be reloaded is not a result. The next group loads each saved checkpoint from disk, checks the model hash, and scores the same held-out rows again: the loss must match the loss that the model had in memory. Then it runs the reloaded policy on development states 10–14 and pairs the outcome with the *before* outcome case by case.

With five cases, the paired change is a pilot signal. It says whether adaptation helped on the development states. It is not used to tune anything.

In [ ]:
AFTER_DIR = OUT_DIR / "dev-after"
METHOD_CASES = [case_id(state, BASE_SEED + state) for state in METHOD_STATES]
reload_report = {}
before_after = {}
for method in METHODS:
    record = ADAPTATIONS[method]
    recipe = START["policies"][method]
    saved_hash = sha256_file(Path(record["checkpoint"]) / "model.safetensors")
    reloaded_loss = checkpoint_heldout_loss(
        method,
        record["checkpoint"],
        configs_root=CONFIGS,
        data_root=DATA,
        settings=ADAPT,
        vlm_dir=VLM_DIR,
    )
    if saved_hash != record["model_sha256"] or not math.isclose(
        reloaded_loss, record["heldout_loss_after"], rel_tol=1e-3, abs_tol=1e-4
    ):
        raise AssertionError(f"{method}: the reloaded checkpoint differs from the trained model")
    after_summary = evaluate_development(
        method,
        record["checkpoint"],
        configs_root=CONFIGS,
        data_root=DATA,
        output_dir=AFTER_DIR / method,
        state_ids=METHOD_STATES,
        vlm_dir=VLM_DIR,
        inference_steps=START["policies"]["diffusion"]["inference_steps"],
    )
    release_gpu_memory()
    rows = {
        label: episode_case_rows(
            read_jsonl(folder / method / "episodes.jsonl"),
            base_seed=BASE_SEED,
            prompt=START["task_prompt"],
            action_steps=recipe["n_action_steps"],
        )
        for label, folder in (("before", BEFORE_DIR), ("after", AFTER_DIR))
    }
    pair = paired_summary(rows["before"], rows["after"], expected_case_ids=METHOD_CASES)
    p_value = exact_sign_test(pair.right_only, pair.left_only)
    reload_report[method] = {"reloaded_heldout_loss": reloaded_loss, "model_sha256": saved_hash}
    before_after[method] = {
        "heldout_loss_before": record["heldout_loss_before"],
        "heldout_loss_after": record["heldout_loss_after"],
        "successes_before": pair.left_successes,
        "successes_after": pair.right_successes,
        "gains": pair.right_only,
        "losses": pair.left_only,
        "sign_test_p": p_value,
        "cases": pair.cases,
        "after_errors": after_summary["policy_errors"] + after_summary["infrastructure_errors"],
    }
    print(
        method,
        "| held-out loss:",
        round(record["heldout_loss_before"], 4),
        "->",
        round(reloaded_loss, 4),
        "| development successes:",
        pair.left_successes,
        "->",
        pair.right_successes,
        "| gains:",
        pair.right_only,
        "| losses:",
        pair.left_only,
        "| p:",
        round(p_value, 4),
    )
(OUT_DIR / "before-after.json").write_text(
    json.dumps(
        {"reload": reload_report, "development": before_after, "states": METHOD_STATES},
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

### The freeze: the new comparison recipe

The recipe for the final comparison now names the **adapted** checkpoints. Everything else is copied from the start recipe: the task, the prompt, the seeds, the time limit, the final-state list, and the execution settings of each policy. The recipe text for each policy records its start checkpoint, its number of adaptation updates, and its pretraining, with the adaptation added. `load_comparison_config` checks that the list of final states is the full reserved list, in order, and `verify_recipe_files` checks every model hash.

The reserved cases are the **final states**, 20 to 49, 30 states in all. The training data and the earlier lessons never used them. Each policy gets one attempt per state, with reset seed `20260930 + state_id`.

The rule for the final states is simple: **use them once, after the freeze, and change nothing after you see a result.** If you tune a recipe on these states, the result no longer measures how the recipe works on new cases. A result with wide uncertainty is a valid result. Do not run the evaluation again to get a better number.

In [ ]:
DRAFT_RECIPE = OUT_DIR / "adapted-recipe-draft.yaml"
DRAFT_RECIPE.write_text(yaml.safe_dump(adapted_recipe(START, ADAPTATIONS), sort_keys=False))
FROZEN_DIR = OUT_DIR / "frozen"
FROZEN_DIR.mkdir()
FROZEN_CONFIG = freeze_comparison_recipe(DRAFT_RECIPE, workspace=WORKSPACE, output_dir=FROZEN_DIR)
COMPARISON = load_comparison_config(FROZEN_CONFIG, CONFIGS)
MODEL_HASHES = verify_recipe_files(COMPARISON)
CONFIG_SHA256 = sha256_file(FROZEN_CONFIG)
PARAMETER_COUNTS = {
    method: count_safetensors_parameters(Path(recipe["checkpoint"]) / "model.safetensors")
    for method, recipe in COMPARISON["policies"].items()
}
if (OUT_DIR / "evaluation").exists():
    raise FileExistsError("A final evaluation already exists in this run directory")
(OUT_DIR / "freeze.json").write_text(
    json.dumps(
        {
            "config": str(FROZEN_CONFIG),
            "config_sha256": CONFIG_SHA256,
            "model_sha256": MODEL_HASHES,
            "parameter_counts": PARAMETER_COUNTS,
            "frozen_unix_s": time.time(),
            "final_states_used_before_freeze": [],
        },
        indent=2,
        sort_keys=True,
    )
    + "\n"
)
print("frozen recipe:", FROZEN_CONFIG)
for method, recipe in COMPARISON["policies"].items():
    print(
        method,
        "| robot training steps:",
        recipe["robot_training_steps"],
        "| adaptation updates:",
        recipe["adaptation_updates"],
        "| n_action_steps:",
        recipe["n_action_steps"],
        "| pretraining:",
        recipe["pretraining"],
    )

### Required gate: verify the completed Lesson 21 run

Before the final states open, the experiment finds the latest completed Lesson 21 run that matches this task, dataset revision, and training-split hash. It checks both training arms, their checkpoint hashes, reload checks, and all five intervention-state evaluations. Set `VLA_LESSON21_RUN` to a specific run directory when you need an exact artifact. The selected run path is printed and saved with the final results. This is an evidence check, not an exercise.


In [ ]:
if sha256_file(FROZEN_CONFIG) != CONFIG_SHA256:
    raise ValueError("The frozen recipe changed after the freeze")
LESSON21_OVERRIDE = os.environ.get("VLA_LESSON21_RUN")
DATA_UNIT_REPORT = find_completed_data_unit_run(
    DATA / "course-results",
    task_alias=COMPARISON["task_alias"],
    dataset_revision=COMPARISON["dataset_revision"],
    train_split_sha256=COMPARISON["train_split_sha256"],
    override_path=None if LESSON21_OVERRIDE is None else Path(LESSON21_OVERRIDE),
)
print("Verified Lesson 21 run:", DATA_UNIT_REPORT["run_dir"])
FINAL_STATES = final_cases(
    data_unit_complete=DATA_UNIT_REPORT["state_ids"] == list(range(15, 20)),
    recipes_frozen=(OUT_DIR / "freeze.json").is_file(),
)
if FINAL_STATES != COMPARISON["state_ids"]:
    raise ValueError("The frozen recipe does not name the full final-state list")
EVALUATION_DIR = OUT_DIR / "evaluation"
EVALUATION_DIR.mkdir()
evaluation_started = time.monotonic()
evaluation_use = {}
for method in METHODS:
    with measure_cuda_use() as use:
        evaluate_frozen_method_timed(
            method=method,
            config_path=str(FROZEN_CONFIG),
            configs_root=str(CONFIGS),
            data_root=str(DATA),
            output_dir=str(EVALUATION_DIR / method),
        )
    release_gpu_memory()
    evaluation_use[method] = {
        "seconds": use.seconds,
        "peak_allocated_mib": use.peak_allocated_mib,
        "peak_reserved_mib": use.peak_reserved_mib,
    }
    summary = json.loads((EVALUATION_DIR / method / "summary.json").read_text())
    print(
        method,
        "| attempts:",
        summary["attempts"],
        "| successes:",
        summary["successes"],
        "| policy errors:",
        summary["policy_errors"],
        "| infrastructure errors:",
        summary["infrastructure_errors"],
        "| seconds:",
        round(use.seconds, 1),
    )
(OUT_DIR / "evaluation-resources.json").write_text(
    json.dumps(
        {
            "methods": evaluation_use,
            "lesson21_run": DATA_UNIT_REPORT["run_dir"],
            "scope": "PyTorch CUDA allocator in this process during policy load and all attempts; excludes other processes",
        },
        indent=2,
        sort_keys=True,
    )
    + "\n"
)
print("evaluation group seconds:", round(time.monotonic() - evaluation_started, 1))

In [ ]:
EXPECTED_CASES = [
    case_id(state, COMPARISON["base_seed"] + state) for state in COMPARISON["state_ids"]
]
rows_by_method = {}
table: list[ComparisonRow] = []
exposure_by_method = {}
training_by_method = {}
time_by_method = {}
for method in METHODS:
    recipe = COMPARISON["policies"][method]
    record = ADAPTATIONS[method]
    records = read_jsonl(EVALUATION_DIR / method / "episodes.jsonl")
    rows = episode_case_rows(
        records,
        base_seed=COMPARISON["base_seed"],
        prompt=COMPARISON["task_prompt"],
        action_steps=recipe["n_action_steps"],
    )
    rows_by_method[method] = rows
    summary = json.loads((EVALUATION_DIR / method / "summary.json").read_text())
    if summary["infrastructure_errors"]:
        raise RuntimeError(
            f"{method}: {summary['infrastructure_errors']} infrastructure errors; "
            f"inspect {EVALUATION_DIR / method / 'episodes.jsonl'} before comparison"
        )
    timing = json.loads((EVALUATION_DIR / method / "inference-timing.json").read_text())
    samples = [
        value
        for attempt in timing["attempts"]
        for value in attempt["action_generation"]["samples_s"]
    ]
    generation = latency_summary(samples)
    successes = sum(row.success for row in rows)
    interval = wilson_interval(successes, len(rows))
    times = episode_time_summary(records)
    time_by_method[method] = times
    table.append(
        {
            "method": method,
            "successes": successes,
            "attempts": len(rows),
            "success_rate": successes / len(rows),
            "wilson_low": interval["low"],
            "wilson_high": interval["high"],
            "policy_errors": summary["policy_errors"],
            "infrastructure_errors": summary["infrastructure_errors"],
            "mean_simulated_episode_s": times["mean_simulated_s"],
            "mean_wall_episode_s": times["mean_wall_s"],
            "generation_calls": generation["count"],
            "generation_mean_s": generation["mean_s"],
            "generation_p95_s": generation["p95_s"],
            "generation_max_s": generation["max_s"],
            "peak_allocated_mib": evaluation_use[method]["peak_allocated_mib"],
            "peak_reserved_mib": evaluation_use[method]["peak_reserved_mib"],
            "robot_training_steps": recipe["robot_training_steps"],
            "adaptation_updates": record["adaptation_updates"],
            "adaptation_wall_s": record["train_seconds"],
            "adaptation_peak_allocated_mib": record["peak_allocated_mib"],
            "n_action_steps": recipe["n_action_steps"],
            "pretraining": str(recipe["pretraining"]),
            "parameters": PARAMETER_COUNTS[method],
        }
    )
    if samples:
        print(method, "real-time:", realtime_report(samples, recipe["n_action_steps"]))
    rate = training_rate(record["metrics_path"], warmup_steps=20)
    training_by_method[method] = rate
    exposure_by_method[method] = training_exposure(
        record["adaptation_updates"], record["batch_size"], 1 / rate["steady_steps_per_s"]
    )
    print(
        method,
        "adaptation:",
        record["adaptation_updates"],
        "updates | samples seen:",
        exposure_by_method[method]["samples_seen"],
        "| steady steps/s:",
        round(rate["steady_steps_per_s"], 3),
        "| hours (measured rate):",
        round(exposure_by_method[method]["projected_hours"], 4),
        "| wall seconds:",
        round(record["train_seconds"], 1),
    )
    print(
        method,
        "simulated episode s:",
        times["mean_simulated_s"],
        "| wall episode s:",
        times["mean_wall_s"],
        "| wall / simulated:",
        times["wall_over_simulated"],
    )

pairs = []
for index, left in enumerate(METHODS):
    for right in METHODS[index + 1 :]:
        pair = paired_summary(
            rows_by_method[left], rows_by_method[right], expected_case_ids=EXPECTED_CASES
        )
        p_value = exact_sign_test(pair.left_only, pair.right_only)
        pairs.append(
            {"left": left, "right": right, "summary": asdict(pair), "sign_test_p": p_value}
        )
        print(
            left,
            "vs",
            right,
            "| only left:",
            pair.left_only,
            "| only right:",
            pair.right_only,
            "| p:",
            round(p_value, 4),
        )
for row in table:
    print(row)

REPORT_DIR = OUT_DIR / "report"
REPORT_DIR.mkdir()
(REPORT_DIR / "table.json").write_text(
    json.dumps(
        {
            "table": table,
            "pairs": pairs,
            "exposure": exposure_by_method,
            "training_rate": training_by_method,
            "adaptations": ADAPTATIONS,
            "development_before_after": before_after,
            "episode_times": time_by_method,
        },
        indent=2,
        sort_keys=True,
    )
    + "\n"
)
figure, axes = plt.subplots(2, 2, figsize=(11, 7))
rates = [row["success_rate"] for row in table]
axes[0, 0].bar(
    METHODS,
    rates,
    yerr=[
        [row["success_rate"] - row["wilson_low"] for row in table],
        [row["wilson_high"] - row["success_rate"] for row in table],
    ],
    capsize=4,
)
axes[0, 0].set_ylim(0, 1)
axes[0, 0].set_ylabel("final success (95 % Wilson)")
axes[0, 1].bar(METHODS, [1000 * (row["generation_p95_s"] or 0.0) for row in table])
axes[0, 1].axhline(1000 / LIBERO_CONTROL_HZ, color="black", linestyle="--")
axes[0, 1].set_ylabel("p95 call time (ms); dashed: 50 ms")
positions = np.arange(len(METHODS))
axes[1, 0].bar(
    positions - 0.2,
    [row["adaptation_peak_allocated_mib"] for row in table],
    0.4,
    label="adaptation",
)
axes[1, 0].bar(
    positions + 0.2, [row["peak_allocated_mib"] for row in table], 0.4, label="evaluation"
)
axes[1, 0].set_xticks(positions, METHODS)
axes[1, 0].set_ylabel("peak PyTorch GPU MiB")
axes[1, 0].legend()
axes[1, 1].bar(
    positions - 0.2,
    [row["mean_simulated_episode_s"] or 0.0 for row in table],
    0.4,
    label="simulated",
)
axes[1, 1].bar(
    positions + 0.2, [row["mean_wall_episode_s"] or 0.0 for row in table], 0.4, label="wall"
)
axes[1, 1].set_xticks(positions, METHODS)
axes[1, 1].set_ylabel("mean seconds per episode")
axes[1, 1].legend()
for axis in axes.flat:
    axis.tick_params(axis="x", labelrotation=15)
figure.tight_layout()
figure.savefig(REPORT_DIR / "comparison.png", dpi=150)
print("saved report under", REPORT_DIR)

## 9. Guided comparison

Write your own answers in your notes. Use only values that your run printed and saved in `report/table.json` and `before-after.json`.

1. **Adaptation.** For each method, give the held-out loss before and after (compare a method only with itself), the development successes before and after, and the number of updates that really finished. Did the deadline stop any run? Did adaptation help on the development states? Five cases are a pilot signal: say what that allows you to claim.
2. **Success.** For each method, give final successes over attempts and the Wilson interval. Do the intervals overlap? Say what overlap means for your conclusion.
3. **Paired cases.** For each pair of methods, list `only left` and `only right`. Read the exact sign-test value. Is the evidence enough to say that one method is better? Why, or why not?
4. **Failures.** Open `episodes.jsonl` for each method. Count `timeout`, `policy_error`, and `infrastructure_error` separately. A non-zero error count means the run is not a clean comparison. Say so in your result.
5. **Speed.** Compare p95 and maximum call time with the 50 ms period, and with $n\cdot50$ ms. Which methods would fit a synchronous controller? Which would fit an asynchronous controller? Then compare simulated seconds with wall seconds per episode. Remember that this simulator waits, so latency did not change success.
6. **Cost.** Compare the updates, samples seen, wall seconds, and steps per second of the three adaptation runs. They had the same updates and batch size, so a difference in time is a difference in the cost of a step. The start checkpoints came from different numbers of earlier steps (200, 2000, 5000) that this lesson did not measure. What can you not say about the total cost of each recipe?
7. **Memory.** Compare the adaptation peak with the evaluation peak. Which method's adaptation peak is closest to the 48 GB limit?
8. **Pretraining.** Which parts of any difference in success can you blame on the architecture? Which part can you not separate from the start checkpoints and their pretraining?
9. **Claim.** Write one sentence in this form: "On `bowl_plate`, with these prepared recipes, this adaptation protocol, and these 30 cases, ... ." Do not write a sentence that says one architecture is better in general.

Keep the run directory with your notes. A failed or invalid run is a result. Do not delete an attempt.

## 10. Optional extensions

* **Change `n`.** In a new run directory, copy the frozen recipe and change `n_action_steps` for one method. Evaluate on development states 10–14, never on the reserved ones. Does the success change? Does the late-tick fraction change as the formula predicts?
* **Change `K`.** For Diffusion Policy, change `inference_steps` in a copy of the recipe. Measure p95 call time and success on development states. Does the call time grow in proportion to `K`?
* **Equal budget.** Adaptation here used equal updates and batch size. Plan a comparison with equal *wall time* instead. What would it cost with your measured step times? You may plan it with `training_exposure`.
* **Prompt test.** Give SmolVLA a paraphrase of the prompt on development states. Does the success change? ACT and Diffusion Policy cannot read the prompt.
* **Bigger sample.** With the same method and a longer case list on development states, how many cases do you need so that the Wilson interval is narrower than ±0.1? Use `wilson_interval` to find out.

## 11. What you learned

* A fair comparison fixes the task, the training data, the adaptation protocol, the cases, the seeds, and the time limit, and counts every attempt.
* A training experiment needs a deadline inside its loop, a minimum number of updates, a saved and reloaded checkpoint, and a held-out check before and after.
* The recipes differ in start checkpoint and pretraining. A success table alone cannot separate these from the architecture.
* One policy call makes a chunk of `n` actions. Latency is a per-call number. The mean hides the tail, so report p95 and the maximum. Simulated time, model-call time, and episode wall time are three different clocks.
* Report an interval for each rate, and use a paired test on the cases where the methods differ. A small sample gives weak evidence.
* Tune on development states. Freeze. Then use the final states once.

[Course README](../../README.md) · [Course index](../index.ipynb)